# BERT Topic Analysis

In [1]:
import re
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix

from simpletransformers.classification import ClassificationModel, ClassificationArgs

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

c:\Users\adria\Downloads\ba-text-mining-master\ba-text-mining-master\.conda\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1) Load 3 datasets

movies, restaurants, and books

In [2]:
from datasets import load_dataset
import pandas as pd
from sklearn.model_selection import train_test_split

#movies
imdb = load_dataset("imdb")
movie_texts = imdb["train"]["text"][:3000]
movie_df = pd.DataFrame({
    "text": movie_texts,
    "label": "movie"
})

#restaurants
yelp = load_dataset("yelp_review_full")
restaurant_texts = yelp["train"]["text"][:3000]
restaurant_df = pd.DataFrame({
    "text": restaurant_texts,
    "label": "restaurant"
})

#books
books = load_dataset("cogsci13/Amazon-Reviews-2023-Books-Review")
book_texts = books["full"]["text"][:3000]
book_df = pd.DataFrame({
    "text": book_texts,
    "label": "book"
})

#all
df = pd.concat([
    movie_df,
    restaurant_df,
    book_df
])

df = df.sample(frac=1, random_state=42)

print(df["label"].value_counts())

book          3000
movie         3000
restaurant    3000
Name: label, dtype: int64


Encode labels to integers for BERT

And create a way to get labels from ids

In [3]:
label2id = {
    "movie": 0,
    "restaurant": 1,
    "book": 2
}

id2label = {
    v: k for k, v in label2id.items()
}

df["labels"] = df["label"].map(label2id)

## 2) Train Test Split

In [4]:
train_df, val_df = train_test_split(
    df,
    test_size=0.1,
    stratify=df["labels"],
    random_state=42
)

## 3) Configure and train BERT

IMPORTANT: multiprocessing should be disabled in evaluation and in general training. Windows + Jupyter = bad outcome

Unless you use Linux or Mac, idk about those...

In [6]:
# Model args for CUDA
model_args = ClassificationArgs()

model_args.overwrite_output_dir = True
model_args.evaluate_during_training = True
model_args.num_train_epochs = 10
model_args.train_batch_size = 64
model_args.learning_rate = 4e-6
model_args.max_seq_length = 128

model_args.use_cached_eval_features = True
model_args.reprocess_input_data = False
# Windows + Jupyter: multiprocessing eval sucks (stuck at 0it). Keep this False.
model_args.use_multiprocessing_for_evaluation = False

model_args.eval_batch_size = 64

steps_per_epoch = 32
model_args.evaluate_during_training_steps = 0
model_args.evaluate_each_epoch = True

model_args.use_early_stopping = True
model_args.early_stopping_delta = 0.01
model_args.early_stopping_metric = "eval_loss"
model_args.early_stopping_metric_minimize = True
model_args.early_stopping_patience = 2

model_args.fp16 = True
model_args.dataloader_num_workers = 0
model_args.use_multiprocessing = False
model_args.save_eval_checkpoints = False
model_args.save_model_every_epoch = False
model_args.evaluate_during_training_silent = True


In [9]:
model = ClassificationModel(
    "bert",
    "bert-base-uncased",
    num_labels=3,
    args=model_args,
    use_cuda=True,
)

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [11]:
model.train_model(
    train_df[["text", "labels"]],
    eval_df=val_df[["text", "labels"]]
)

c:\Users\adria\Downloads\ba-text-mining-master\ba-text-mining-master\.conda\Lib\site-packages\simpletransformers\classification\classification_model.py:544: UserWarning: The 'eval_df' parameter has been deprecated and will be removed in a future version. Please use 'eval_data' instead.
  warnings.warn(
Epoch:   0%|          | 0/10 [00:00<?, ?it/s]c:\Users\adria\Downloads\ba-text-mining-master\ba-text-mining-master\.conda\Lib\site-packages\simpletransformers\classification\classification_model.py:924: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = amp.GradScaler()
Epoch 1 of 10:   0%|          | 0/10 [00:00<?, ?it/s]c:\Users\adria\Downloads\ba-text-mining-master\ba-text-mining-master\.conda\Lib\site-packages\simpletransformers\classification\classification_model.py:950: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with

(1270,
 defaultdict(list,
             {'global_step': [127,
               254,
               381,
               508,
               635,
               762,
               889,
               1016,
               1143,
               1270],
              'train_loss': [0.3139784038066864,
               0.09684710949659348,
               0.06450971215963364,
               0.020665062591433525,
               0.031809065490961075,
               0.007202254608273506,
               0.005394935607910156,
               0.004564391449093819,
               0.00750732421875,
               0.004794385749846697],
              'mcc': [0.9517988701356197,
               0.9750668124224664,
               0.9817339359262097,
               0.9816684845729511,
               0.985,
               0.983335154326046,
               0.9850127687668008,
               0.986668493832236,
               0.986668493832236,
               0.986668493832236],
              'eval_loss': [0.3168664

## 5) Predict topics on Sentiment-topic-test.tsv

In [ ]:
st_df = pd.read_csv("Sentiment-topic-test.tsv", sep="\t")
texts = st_df["text"].astype(str).tolist()

#without this it tries to use cached data, which would not fit test data
_prev_eval_bs = model.args.eval_batch_size
_prev_use_cache = model.args.use_cached_eval_features
_prev_no_cache = model.args.no_cache
_prev_reprocess = model.args.reprocess_input_data

model.args.use_cached_eval_features = False
model.args.no_cache = True
model.args.reprocess_input_data = True
model.args.eval_batch_size = len(texts)

pred_ids, raw_outputs = model.predict(texts)

for i, row in enumerate(st_df.itertuples(index=False)):
    pred_topic = id2label[int(pred_ids[i])]
    correct = pred_topic == row.topic
    print(f"{'Y' if correct else 'N'} {row.topic}/{pred_topic} {row.text}")

Predicting:   0%|          | 0/1 [00:00<?, ?it/s]c:\Users\adria\Downloads\ba-text-mining-master\ba-text-mining-master\.conda\Lib\site-packages\simpletransformers\classification\classification_model.py:2260: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with amp.autocast():
Predicting: 100%|██████████| 1/1 [00:00<00:00,  6.02it/s]

Y movie/movie It took eight years for Warner Brothers to recover from the disaster that was this movie.
Y restaurant/restaurant All the New York University students love this diner in Soho so it makes for a fun young atmosphere.
Y restaurant/restaurant This Italian place is really trendy but they have forgotten about the most important part of a restaurant, the food.
Y book/book In conclusion, my review of this book would be: I like Jane Austen and understand why she is famous.
Y movie/movie The story of this movie is focused on Carl Brashear played by Cuba Gooding Jr. who wants to be the first African American deep sea diver in the Navy.
Y movie/movie Chris O'Donnell stated that while filming for this movie, he felt like he was in a toy commercial.
Y restaurant/restaurant My husband and I moved to Amsterdam 6 years ago and for as long as we have lived here, Blauwbrug has been our favorite place to eat!
Y movie/movie Dame Maggie Smith performed her role excellently, as she does in all 